# Exam AI: one base model, one LoRA adapter per exam family (Kaggle, free GPU)

This notebook trains the exam tutor planned in [`ops/plan/exam-llm.md`](../../ops/plan/exam-llm.md):
one permissively licensed small base model, a QLoRA adapter per exam family, trained only on content this
project has the right to use.

**Every model and dataset, with its license** (checked against the official page on 2026-09-28; the machine-readable
copy is [`licenses.json`](licenses.json) and the notebook refuses to train if any of them lacks a license):

| Used | Model / dataset | License | Verified against |
|---|---|---|---|
| base | `Qwen/Qwen2.5-1.5B-Instruct` | Apache-2.0 | https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct |
| alternative | `HuggingFaceTB/SmolLM2-1.7B-Instruct` | Apache-2.0 | https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B-Instruct |
| rejected | `PhysicsWallahAI/Aryabhata-1.0` | CC BY-NC 4.0 (blocks commercial use) | https://huggingface.co/PhysicsWallahAI/Aryabhata-1.0 |
| data | this list's pages, modules and formula sheets | CC BY-SA 4.0 | [`LICENSE.md`](../../LICENSE.md) |
| data | this list's original questions (`questions/`) | CC BY-SA 4.0 | [`questions/README.md`](../../questions/README.md) |
| replay | `HuggingFaceH4/ultrachat_200k` (small capped sample) | MIT | https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k |

**The release rule, enforced by code:** a family adapter is published only if it *beats* the base model on the
held-out T-122 evaluation set (`questions/*-eval-*.json`, 50 questions). The notebook writes `results.json`,
recomputes the scores with `scripts/lora_release_gate.py`, and raises instead of releasing when the adapter does
not win. The `*-eval-*` files are never trained on.

## How to run (free, ~45-90 min)

1. Kaggle -> New Notebook -> File -> Import Notebook -> upload this file.
2. Settings -> Accelerator -> **GPU T4 x2** (or P100), Internet -> **on**.
3. Optionally attach this repository as a Kaggle dataset named `awesome-indian-exams` (it is uploaded whole);
   otherwise the notebook clones the public GitHub repository.
4. Run All. Outputs land in `/kaggle/working/` (`results.json`, `adapters/`, `release/` only if the gate passes).
   T-124 publishes the `release/` folder to Hugging Face by hand, never on a schedule.

Runtime note: Kaggle gives about 30 GPU hours a week; this run uses a fraction of it. Nothing here calls a
paid API.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/lakhidas168-ship-it/awesome-indian-exams.git"
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd() / "lora-work"
WORK.mkdir(parents=True, exist_ok=True)


def find_repo() -> Path:
    """Use the attached dataset/clone when present, otherwise clone the public repository."""
    for base in (Path("/kaggle/input"), Path.cwd(), Path("/content")):
        if not base.is_dir():
            continue
        for marker in sorted(base.rglob("notebooks/lora/licenses.json")):
            repo = marker.parents[2]
            if (repo / "registry" / "exams.toml").exists():
                return repo.resolve()
    repo = WORK / "awesome-indian-exams"
    if not (repo / "notebooks" / "lora" / "licenses.json").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
    return repo


REPO = find_repo()
sys.path.insert(0, str(REPO / "scripts"))
print("repository:", REPO)
print("working dir:", WORK)


In [ ]:
PACKAGES = ["transformers>=4.44,<6", "peft>=0.12,<1", "bitsandbytes>=0.43", "accelerate>=0.33"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)
print("installed:", ", ".join(PACKAGES))


In [ ]:
import lora_data
import lora_release_gate as gate

REGISTRY_PATH = REPO / "notebooks" / "lora" / "licenses.json"
REGISTRY = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
license_problems = gate.registry_problems(REGISTRY)
if license_problems:
    raise SystemExit("license registry is incomplete, refusing to train:\n- " + "\n- ".join(license_problems))
BASE_ID = gate.used_base_model(REGISTRY)
USED_DATASETS = [d for d in REGISTRY["datasets"] if d.get("used") is True]
base_entry = next(m for m in REGISTRY["base_models"] if m["id"] == BASE_ID)
print(f"base model: {BASE_ID}  license: {base_entry['license']}  verified: {base_entry['verified']}")
for dataset in USED_DATASETS:
    print(f"dataset: {dataset['id']:<44} {dataset['license']:<12} {dataset['source_url']}")
print(f"output license: {REGISTRY['output_license']['name']}")


In [ ]:
MIN_EXAMPLES = 8
REPLAY_EXAMPLES = 2000
MAX_LENGTH = 768

questions = lora_data.load_questions(REPO / "questions")
train_questions, eval_questions = lora_data.split_train_eval(questions)
exam_family = lora_data.load_exam_families(REPO / "registry" / "exams.toml")
family_examples = lora_data.build_all_family_examples(REPO, train_questions, exam_family)
eval_families = sorted({f for q in eval_questions.values()
                        for f in lora_data.question_families(q, exam_family)})

print(f"training questions (eval sets excluded): {len(train_questions)}")
print(f"held-out T-122 evaluation questions: {len(eval_questions)}")
print("families with an evaluation slice:", ", ".join(eval_families) or "(none)")
for family in sorted(family_examples):
    count = len(family_examples[family])
    verdict = "train" if count >= MIN_EXAMPLES else "skip (too few examples)"
    print(f"  {family:<24} {count:>3} examples  {verdict}")


In [ ]:
import gc
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments

tokenizer = AutoTokenizer.from_pretrained(BASE_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


class SFTDataset(torch.utils.data.Dataset):
    """Prompt-completion rows: the loss only counts the assistant's answer."""

    def __init__(self, examples, tokenizer, max_length):
        self.rows = []
        for example in examples:
            messages = example["messages"]
            prompt = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
            completion = messages[-1]["content"] + tokenizer.eos_token
            prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
            completion_ids = tokenizer(completion, add_special_tokens=False)["input_ids"][:max_length]
            input_ids = (prompt_ids + completion_ids)[-max_length:]
            labels = ([-100] * len(prompt_ids) + completion_ids)[-max_length:]
            self.rows.append({"input_ids": input_ids, "labels": labels})

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index]


class PadCollator:
    def __init__(self, pad_id):
        self.pad_id = pad_id

    def __call__(self, batch):
        length = max(len(row["input_ids"]) for row in batch)
        input_ids, labels, attention = [], [], []
        for row in batch:
            pad = length - len(row["input_ids"])
            input_ids.append(row["input_ids"] + [self.pad_id] * pad)
            labels.append(row["labels"] + [-100] * pad)
            attention.append([1] * len(row["input_ids"]) + [0] * pad)
        return {"input_ids": torch.tensor(input_ids), "labels": torch.tensor(labels),
                "attention_mask": torch.tensor(attention)}


def load_base():
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    return AutoModelForCausalLM.from_pretrained(BASE_ID, quantization_config=quant, device_map="auto")


def replay_examples():
    """A small capped sample of the MIT-licensed replay set, so general answers survive."""
    try:
        stream = load_dataset("HuggingFaceH4/ultrachat_200k", split=f"train_sft[:{REPLAY_EXAMPLES}]")
    except Exception as exc:
        print("replay dataset unavailable, continuing without it:", exc)
        return []
    examples = []
    for row in stream:
        messages = [{"role": m["role"], "content": m["content"]}
                    for m in row["messages"] if m.get("role") in ("user", "assistant")]
        for index in range(len(messages) - 1, 0, -1):
            if messages[index]["role"] == "assistant" and messages[index - 1]["role"] == "user":
                examples.append({"messages": [{"role": "system", "content": lora_data.SYSTEM_PROMPT},
                                              messages[index - 1], messages[index]]})
                break
    return examples


REPLAY = replay_examples()
print("replay examples:", len(REPLAY))


def train_adapter(family, examples):
    model = prepare_model_for_kbit_training(load_base())
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                      "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(model, lora)
    args = TrainingArguments(output_dir=str(WORK / "train" / family),
                             per_device_train_batch_size=4, gradient_accumulation_steps=4,
                             num_train_epochs=2, learning_rate=2e-4, lr_scheduler_type="cosine",
                             warmup_ratio=0.03, logging_steps=10, save_strategy="no",
                             report_to=[], fp16=torch.cuda.is_available(), seed=42)
    trainer = Trainer(model=model, args=args,
                      train_dataset=SFTDataset(examples, tokenizer, MAX_LENGTH),
                      data_collator=PadCollator(tokenizer.pad_token_id))
    trainer.train()
    out = WORK / "adapters" / family
    model.save_pretrained(str(out))
    tokenizer.save_pretrained(str(out))
    del model, trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out


ADAPTERS = {}
for family in sorted(family_examples):
    examples = family_examples[family]
    if len(examples) < MIN_EXAMPLES:
        print(f"{family}: skipped, only {len(examples)} examples")
        continue
    extra = REPLAY if family in eval_families else []
    ADAPTERS[family] = train_adapter(family, examples + extra)
    print(f"{family}: adapter saved to {ADAPTERS[family]}")


In [ ]:
import datetime as dt
from peft import PeftModel

EVAL_MAX_NEW_TOKENS = 160


def build_eval_prompt(question):
    options = question.get("options") or []
    letters = "\n".join(f"{lora_data.LETTERS[i]}. {option}" for i, option in enumerate(options))
    return (f"Question: {question['question']}\nOptions:\n{letters}\n"
            "Answer with the letter of one option. If the list cannot answer, choose the Not sure option. "
            "Reply with the letter on the first line as 'ANSWER: <letter>', then REASON and SOURCE.")


def generate_answers(model, questions, max_new_tokens=EVAL_MAX_NEW_TOKENS):
    model.eval()
    predictions = {}
    for qid in sorted(questions):
        question = questions[qid]
        messages = [{"role": "system", "content": lora_data.SYSTEM_PROMPT},
                    {"role": "user", "content": build_eval_prompt(question)}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
        reply = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        predictions[qid] = lora_data.score_prediction(reply, question.get("options") or [])
    return predictions


def block(predictions):
    return {"predictions": predictions, **lora_data.summarise(eval_questions, predictions)}


base_model = load_base()
base_predictions = generate_answers(base_model, eval_questions)
results = {
    "schema": gate.RESULTS_SCHEMA,
    "base_model": BASE_ID,
    "eval_set": "questions/*-eval-*.json (held out, never trained on)",
    "created": dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "base": block(base_predictions),
    "adapters": {},
}
print(f"base: {results['base']['correct']}/{results['base']['total']} correct "
      f"({results['base']['accuracy']:.2f}), probes {results['base']['probe_correct']}/{results['base']['probe_total']}")

peft_model = None
for family in eval_families:
    if family not in ADAPTERS:
        continue
    if peft_model is None:
        peft_model = PeftModel.from_pretrained(base_model, str(ADAPTERS[family]))
    else:
        peft_model.load_adapter(str(ADAPTERS[family]), adapter_name=family)
        peft_model.set_adapter(family)
    predictions = generate_answers(peft_model, eval_questions)
    summary = lora_data.summarise(eval_questions, predictions)
    beats = summary["correct"] > results["base"]["correct"]
    reason = ("" if beats else f"adapter {summary['correct']}/{summary['total']} did not beat "
                                f"the base {results['base']['correct']}/{results['base']['total']}")
    results["adapters"][family] = {"predictions": predictions, **summary, "released": beats, "reason": reason}
    print(f"{family}: {summary['correct']}/{summary['total']} correct "
          f"({summary['accuracy']:.2f}), beats base: {beats}")

primary = "engineering" if "engineering" in results["adapters"] else (
    sorted(results["adapters"])[0] if results["adapters"] else "")
primary_correct = results["adapters"].get(primary, {}).get("correct")
allowed = primary_correct is not None and primary_correct > results["base"]["correct"]
results["release"] = {
    "allowed": bool(allowed),
    "primary": primary,
    "reason": (f"{primary} adapter {primary_correct}/{results['adapters'][primary]['total']} vs base "
               f"{results['base']['correct']}/{results['base']['total']} on the T-122 set") if allowed
              else "no adapter beat the base model on the T-122 set, nothing is released",
}


In [ ]:
import shutil

problems = gate.results_problems(results, eval_questions, REGISTRY)
if problems:
    raise SystemExit("results did not pass the release gate:\n- " + "\n- ".join(problems))

results_path = WORK / "results.json"
results_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
print("wrote", results_path)

allowed, reasons = gate.release_allowed(results, eval_questions)
if not allowed:
    raise SystemExit("release blocked: " + "; ".join(reasons) + "\n"
                     "The T-122 scores are in results.json; improve the data or training and run again.")

release_dir = WORK / "release"
released = []
for family, block in sorted(results["adapters"].items()):
    if not block.get("released"):
        continue
    target = release_dir / family
    shutil.copytree(ADAPTERS[family], target, dirs_exist_ok=True)
    (target / "MODEL_CARD.md").write_text(gate.render_model_card(REGISTRY, results, family), encoding="utf-8")
    released.append(family)
    print(f"released {family} adapter -> {target}")
print("released families:", ", ".join(released))


## What this notebook does not claim

- The T-122 set has 50 engineering-family questions. It is a floor, not a benchmark: no adapter is released for a
  family whose evaluation slice does not exist yet (those adapters stay as candidates in `/kaggle/working/adapters/`).
- Exact dates, cutoffs and vacancy counts are deliberately *not* learned as facts; the training data teaches the
  "Not sure - check the official notification" answer with the official link, and the website retrieves facts from
  the list's pages at answer time (`T-120`/`T-121`).
- The adapter, its results and the model card are the artifacts T-124 publishes by hand with the owner's token.

## Sources

- Base/alternative/rejected model licenses: the Hugging Face model pages listed in the first cell, fetched 2026-09-28.
- Dataset licenses: [`licenses.json`](licenses.json); content under the repository's [`LICENSE.md`](../../LICENSE.md).
- Plan and method: [`ops/plan/exam-llm.md`](../../ops/plan/exam-llm.md).
- Gate and tests: [`scripts/lora_release_gate.py`](../../scripts/lora_release_gate.py),
  [`tests/test_lora_release_gate.py`](../../tests/test_lora_release_gate.py).
